# GPT-5 Evaluation on `ChanceFocus/flare-finred`

This notebook evaluates GPT-5 on the Hugging Face dataset `ChanceFocus/flare-finred`.

**Task:** financial relation extraction. For each sentence, extract all `(head, tail, relation)` triplets.

**Dataset columns used:** `id`, `query`, `text`, `answer`, `label`.

**Evaluation protocol**
- Use the official `test` split.
- Ask GPT-5 to return structured JSON with a `triplets` list.
- Compare predicted triplets against the gold `label` triplets.
- Report strict normalized triplet-level Precision / Recall / F1, exact-match rate, and per-relation metrics.
- Save detailed predictions for error analysis.

> Recommended: run with `LIMIT = 10` first, then set `LIMIT = None` for the full evaluation.


In [12]:
!pip -q install -U --upgrade-strategy only-if-needed openai datasets tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 121.2 MB/s eta 0:00:00


In [13]:
import os
import re
import ast
import json
import time
from pathlib import Path
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from openai import OpenAI
from sklearn.metrics import classification_report

## 1. Configure API and model

In Colab, store your key as a secret named `OPENAI_API_KEY`, or set it in your environment before running the notebook.


In [14]:
# Recommended for Colab:
# Left sidebar → Secrets → add OPENAI_API_KEY → enable notebook access

try:
    from google.colab import userdata
    key_from_colab = userdata.get("OPENAI_API_KEY")
    if key_from_colab:
        os.environ["OPENAI_API_KEY"] = key_from_colab
except Exception:
    pass

assert os.environ.get("OPENAI_API_KEY"), "Please set OPENAI_API_KEY first."

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [15]:
MODEL = "gpt-5.5"
REASONING_EFFORT = "low"

DATASET_NAME = "ChanceFocus/flare-finred"
SPLIT = "test"

# Start small first.
# After confirming it works, change LIMIT = None to run all 1068 rows.
LIMIT = 20

OUTPUT_DIR = Path("gpt55_flare_finred_outputs_fixed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL)
print("Reasoning effort:", REASONING_EFFORT)
print("Dataset:", DATASET_NAME)
print("Limit:", LIMIT)

Model: gpt-5.5
Reasoning effort: low
Dataset: ChanceFocus/flare-finred
Limit: 20


In [16]:
dataset = load_dataset(DATASET_NAME)

print(dataset)
print("Available splits:", list(dataset.keys()))

for split in dataset:
    print(split, len(dataset[split]), dataset[split].column_names)

df_all = dataset[SPLIT].to_pandas()
df_all.head()

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'text', 'answer', 'label'],
        num_rows: 1068
    })
})
Available splits: ['test']
test 1068 ['id', 'query', 'text', 'answer', 'label']


,id,query,text,answer,label
0,finred0,"Given the following sentence, identify the hea...","Wednesday, July 8, 2015 10:30AM IST (5:00AM GM...",PeopleSoft ; JD Edwards ; subsidiary,[PeopleSoft ; JD Edwards ; subsidiary]
1,finred1,"Given the following sentence, identify the hea...",The Daily Show with Trevor Noah premieres toni...,VH1 ; Viacom ; owned_by,[VH1 ; Viacom ; owned_by]
2,finred2,"Given the following sentence, identify the hea...","""Our results for the quarter show very balance...",Michael Corbat ; Citigroup ; employer,[Michael Corbat ; Citigroup ; employer]
3,finred3,"Given the following sentence, identify the hea...","Saudi Arabian budget carrier flynas, which mad...",Airbus ; aircraft ; product_or_material_produced,[Airbus ; aircraft ; product_or_material_produ...
4,finred4,"Given the following sentence, identify the hea...",First Eagle is currently owned by members of t...,TA Associates ; private equity ; industry,[TA Associates ; private equity ; industry]


In [17]:
def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,.;:!?%)\]])", r"\1", s)
    s = re.sub(r"([%(\[])\s+", r"\1", s)
    return s


def normalize_entity(s: str) -> str:
    s = clean_text(s)
    s = s.strip(" \"'`")
    return s


def normalize_relation(s: str) -> str:
    return clean_text(s).strip(" \"'`").lower()


def normalize_triplet(triplet):
    head, tail, relation = triplet
    return (
        normalize_entity(head),
        normalize_entity(tail),
        normalize_relation(relation),
    )


def parse_gold_triplets(label_value):
    """
    FinRED label examples:
    ["PeopleSoft ; JD Edwards ; subsidiary"]

    Each item is:
    head ; tail ; relation
    """
    if label_value is None:
        return []

    if isinstance(label_value, str):
        try:
            parsed = ast.literal_eval(label_value)
            if isinstance(parsed, list):
                label_value = parsed
            else:
                label_value = [label_value]
        except Exception:
            label_value = [label_value]

    if not isinstance(label_value, list):
        label_value = list(label_value)

    triplets = []

    for item in label_value:
        if item is None:
            continue

        item = clean_text(item)

        if not item:
            continue

        parts = [p.strip() for p in item.split(";")]

        if len(parts) != 3:
            continue

        head, tail, relation = parts
        triplets.append(normalize_triplet((head, tail, relation)))

    return triplets


# Test parsing
for i in range(5):
    print("Raw label:", df_all.iloc[i]["label"])
    print("Parsed:", parse_gold_triplets(df_all.iloc[i]["label"]))
    print()

Raw label: ['PeopleSoft ; JD Edwards ; subsidiary']
Parsed: [('PeopleSoft', 'JD Edwards', 'subsidiary')]

Raw label: ['VH1 ; Viacom ; owned_by']
Parsed: [('VH1', 'Viacom', 'owned_by')]

Raw label: ['Michael Corbat ; Citigroup ; employer']
Parsed: [('Michael Corbat', 'Citigroup', 'employer')]

Raw label: ['Airbus ; aircraft ; product_or_material_produced']
Parsed: [('Airbus', 'aircraft', 'product_or_material_produced')]

Raw label: ['TA Associates ; private equity ; industry']
Parsed: [('TA Associates', 'private equity', 'industry')]



In [18]:
# Important:
# Do NOT extract relation labels from query using regex.
# Only extract real relation labels from gold labels.

ALLOWED_RELATIONS = sorted({
    normalize_triplet(t)[2]
    for label in df_all["label"]
    for t in parse_gold_triplets(label)
})

print("Number of allowed relations:", len(ALLOWED_RELATIONS))
print(ALLOWED_RELATIONS)

Number of allowed relations: 29
['brand', 'business_division', 'chairperson', 'chief_executive_officer', 'creator', 'currency', 'developer', 'director_/_manager', 'distributed_by', 'distribution_format', 'employer', 'founded_by', 'headquarters_location', 'industry', 'legal_form', 'location_of_formation', 'manufacturer', 'member_of', 'operator', 'original_broadcaster', 'owned_by', 'owner_of', 'parent_organization', 'platform', 'position_held', 'product_or_material_produced', 'publisher', 'stock_exchange', 'subsidiary']


In [19]:
def old_extract_relations_from_query(query: str):
    if not isinstance(query, str):
        return []
    return re.findall(r"'([^']+)'", query)


bad_relations_from_query = sorted({
    r
    for q in df_all["query"]
    for r in old_extract_relations_from_query(q)
})

print("Old regex relations from query:", len(bad_relations_from_query))
print(bad_relations_from_query[:50])

print()
print("Correct relations from gold labels:", len(ALLOWED_RELATIONS))
print(ALLOWED_RELATIONS)

Old regex relations from query: 84
[' only integrated telco, able to deliver relevant services to our people during this pandemic, sustaining them by keeping them connected, whether they', ' position backed by the leftwing government in Sunday', ' shares helped stem the Nasdaq', 'B', 'Increase the talent pool', 'Mobile banking may overtake internet banking in 2-3 years', 'UAE Domestic Cash Management Bank of the Year', 'brand', 'business_division', 'buy', 'chairperson', 'chief_executive_officer', 'creator', 'currency', 'developer', 'director_/_manager', 'distributed_by', 'distribution_format', 'employer', 'founded_by', 'head ; tail ; rel', 'headquarters_location', 'hold', 'industry', 'legal_form', 'll still have those on the edge of town, in Chicago and South Dallas, South Atlanta, but now they', 'location_of_formation', 'm extremely excited for my next opportunity, I will always be watching my Mylan and Viatris colleagues, cheering them on as they continue to reshape the global health

In [20]:
def make_response_schema(allowed_relations):
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "triplets": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "head": {
                            "type": "string",
                            "description": "The head entity copied from the sentence."
                        },
                        "tail": {
                            "type": "string",
                            "description": "The tail entity copied from the sentence."
                        },
                        "relation": {
                            "type": "string",
                            "enum": allowed_relations,
                            "description": "The relation label."
                        }
                    },
                    "required": ["head", "tail", "relation"]
                }
            }
        },
        "required": ["triplets"]
    }


RESPONSE_SCHEMA = make_response_schema(ALLOWED_RELATIONS)
print(json.dumps(RESPONSE_SCHEMA, indent=2)[:2000])

{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "triplets": {
      "type": "array",
      "items": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "head": {
            "type": "string",
            "description": "The head entity copied from the sentence."
          },
          "tail": {
            "type": "string",
            "description": "The tail entity copied from the sentence."
          },
          "relation": {
            "type": "string",
            "enum": [
              "brand",
              "business_division",
              "chairperson",
              "chief_executive_officer",
              "creator",
              "currency",
              "developer",
              "director_/_manager",
              "distributed_by",
              "distribution_format",
              "employer",
              "founded_by",
              "headquarters_location",
              "industry",
      

In [21]:
def extract_output_text(response):
    """
    New OpenAI Python SDK usually supports response.output_text.
    This fallback makes the notebook more robust.
    """
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text

    try:
        chunks = []
        for item in response.output:
            if hasattr(item, "content"):
                for c in item.content:
                    if hasattr(c, "text"):
                        chunks.append(c.text)
        return "\n".join(chunks)
    except Exception:
        return str(response)


def build_prompt(row):
    allowed_relation_text = "\n".join(f"- {r}" for r in ALLOWED_RELATIONS)

    return f"""
You are evaluating a financial relation extraction dataset.

Your task is to extract relation triplets from the sentence.

Original dataset instruction:
{row["query"]}

Sentence:
{row["text"]}

Allowed relation labels:
{allowed_relation_text}

Return only triplets that match the dataset style.

Each triplet must contain:
- head: the head entity from the sentence
- tail: the tail entity from the sentence
- relation: one of the allowed relation labels

If there is no valid relation, return an empty list.

Do not invent entities.
Do not invent relation labels.
Use the exact allowed relation label when possible.
"""


def predict_triplets(row, max_retries=3):
    prompt = build_prompt(row)

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=MODEL,
                input=[
                    {
                        "role": "system",
                        "content": (
                            "You are a careful financial relation extraction evaluator. "
                            "Return only valid JSON that matches the schema."
                        ),
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                reasoning={"effort": REASONING_EFFORT},
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "finred_relation_extraction",
                        "schema": RESPONSE_SCHEMA,
                        "strict": True,
                    }
                },
            )

            output_text = extract_output_text(response)
            parsed = json.loads(output_text)

            pred_triplets = []
            for t in parsed.get("triplets", []):
                head = t.get("head", "")
                tail = t.get("tail", "")
                relation = t.get("relation", "")
                pred_triplets.append(normalize_triplet((head, tail, relation)))

            # remove duplicates
            pred_triplets = sorted(set(pred_triplets))
            return pred_triplets, output_text, None

        except Exception as e:
            wait = 2 ** attempt
            print(f"Attempt {attempt + 1}/{max_retries} failed: {e}. Waiting {wait}s...")
            time.sleep(wait)

    return [], "", str(e)

In [22]:
row = df_all.iloc[0]

gold = parse_gold_triplets(row["label"])
pred, raw, err = predict_triplets(row)

print("ID:", row["id"])
print("Text:", row["text"])
print("Gold:", gold)
print("Pred:", pred)
print("Error:", err)
print("Raw output:", raw)

ID: finred0
Text: Wednesday, July 8, 2015 10:30AM IST (5:00AM GMT) Rimini Street Comment on Oracle Litigation Las Vegas, United States Rimini Street, Inc., the leading independent provider of enterprise software support for SAP AG’s (NYSE:SAP) Business Suite and BusinessObjects software and Oracle Corporation’s (NYSE:ORCL) Siebel , PeopleSoft , JD Edwards , E-Business Suite , Oracle Database , Hyperion and Oracle Retail software, today issued a statement on the Oracle litigation.
Gold: [('PeopleSoft', 'JD Edwards', 'subsidiary')]
Pred: [('Oracle Corporation', 'E-Business Suite', 'product_or_material_produced'), ('Oracle Corporation', 'Hyperion', 'product_or_material_produced'), ('Oracle Corporation', 'JD Edwards', 'product_or_material_produced'), ('Oracle Corporation', 'NYSE', 'stock_exchange'), ('Oracle Corporation', 'Oracle Database', 'product_or_material_produced'), ('Oracle Corporation', 'Oracle Retail software', 'product_or_material_produced'), ('Oracle Corporation', 'PeopleSoft',

In [23]:
def safe_div(num, den, default=0.0):
    return num / den if den else default


def score_triplet_sets(gold_set, pred_set):
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": gold_set == pred_set,
    }


def summarize_scores(records):
    total_tp = sum(r["tp"] for r in records)
    total_fp = sum(r["fp"] for r in records)
    total_fn = sum(r["fn"] for r in records)

    precision = safe_div(total_tp, total_tp + total_fp)
    recall = safe_div(total_tp, total_tp + total_fn)
    f1 = safe_div(2 * precision * recall, precision + recall)

    exact_match_rate = safe_div(
        sum(1 for r in records if r["exact_match"]),
        len(records)
    )

    api_error_count = sum(1 for r in records if r.get("api_error"))

    return {
        "model": MODEL,
        "dataset": DATASET_NAME,
        "split": SPLIT,
        "reasoning_effort": REASONING_EFFORT,
        "n_examples": len(records),
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match_rate": exact_match_rate,
        "api_error_count": api_error_count,
    }

In [25]:
import numpy as np

def to_jsonable(x):
    """
    Convert numpy / pandas objects into JSON-serializable Python objects.
    """
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_)):
        return bool(x)
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, list):
        return [to_jsonable(i) for i in x]
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    return x


df = df_all.copy()

if LIMIT is not None:
    df = df.head(LIMIT).copy()

safe_model_name = MODEL.replace("/", "_").replace(".", "-")

jsonl_path = OUTPUT_DIR / f"{safe_model_name}_{SPLIT}_flare_finred_predictions.jsonl"
csv_path = OUTPUT_DIR / f"{safe_model_name}_{SPLIT}_flare_finred_predictions.csv"
summary_path = OUTPUT_DIR / f"{safe_model_name}_{SPLIT}_flare_finred_summary.json"
per_relation_path = OUTPUT_DIR / f"{safe_model_name}_{SPLIT}_flare_finred_per_relation.csv"
errors_path = OUTPUT_DIR / f"{safe_model_name}_{SPLIT}_flare_finred_errors.csv"

# Resume from existing JSONL if available
completed = {}

if jsonl_path.exists():
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                completed[str(rec["id"])] = rec

print(f"Evaluating {len(df)} rows. Already completed: {len(completed)}")

records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    row_id = str(row["id"])

    if row_id in completed:
        records.append(completed[row_id])
        continue

    gold_triplets = parse_gold_triplets(row["label"])
    pred_triplets, raw_output, api_error = predict_triplets(row)

    gold_set = set(gold_triplets)
    pred_set = set(pred_triplets)

    score = score_triplet_sets(gold_set, pred_set)

    record = {
        "id": row_id,
        "query": clean_text(row["query"]),
        "text": clean_text(row["text"]),
        "answer": to_jsonable(row.get("answer", "")),
        "label": to_jsonable(row["label"]),

        # Convert tuple triplets into lists for JSON saving
        "gold_triplets": [list(t) for t in gold_triplets],
        "pred_triplets": [list(t) for t in pred_triplets],

        "raw_output": raw_output,
        "api_error": api_error,

        "tp": int(score["tp"]),
        "fp": int(score["fp"]),
        "fn": int(score["fn"]),
        "precision": float(score["precision"]),
        "recall": float(score["recall"]),
        "f1": float(score["f1"]),
        "exact_match": bool(score["exact_match"]),
    }

    with open(jsonl_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    records.append(record)

summary = summarize_scores(records)

print("Summary:")
print(json.dumps(summary, indent=2))

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

pred_df_export = pd.DataFrame(records)
pred_df_export.to_csv(csv_path, index=False)

print()
print("Saved:")
print(csv_path)
print(summary_path)
print(jsonl_path)

Evaluating 20 rows. Already completed: 0


  0%|          | 0/20 [00:00<?, ?it/s]

Summary:
{
  "model": "gpt-5.5",
  "dataset": "ChanceFocus/flare-finred",
  "split": "test",
  "reasoning_effort": "low",
  "n_examples": 20,
  "tp": 2,
  "fp": 46,
  "fn": 20,
  "precision": 0.041666666666666664,
  "recall": 0.09090909090909091,
  "f1": 0.05714285714285715,
  "exact_match_rate": 0.05,
  "api_error_count": 0
}

Saved:
gpt55_flare_finred_outputs_fixed/gpt-5-5_test_flare_finred_predictions.csv
gpt55_flare_finred_outputs_fixed/gpt-5-5_test_flare_finred_summary.json
gpt55_flare_finred_outputs_fixed/gpt-5-5_test_flare_finred_predictions.jsonl


In [ ]:
relation_counts = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for rec in records:
    gold_set = set(tuple(x) for x in rec["gold_triplets"])
    pred_set = set(tuple(x) for x in rec["pred_triplets"])

    for t in gold_set & pred_set:
        relation_counts[t[2]]["tp"] += 1

    for t in pred_set - gold_set:
        relation_counts[t[2]]["fp"] += 1

    for t in gold_set - pred_set:
        relation_counts[t[2]]["fn"] += 1

relation_rows = []

for relation, c in relation_counts.items():
    tp = c["tp"]
    fp = c["fp"]
    fn = c["fn"]

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)

    relation_rows.append({
        "relation": relation,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": tp + fn,
    })

relation_df = pd.DataFrame(relation_rows).sort_values(
    ["f1", "support"],
    ascending=[False, False]
)

relation_df.to_csv(per_relation_path, index=False)
relation_df.head(30)

In [ ]:
error_df = pred_df_export.sort_values(
    ["f1", "exact_match", "id"],
    ascending=[True, True, True]
).head(30)

error_df.to_csv(errors_path, index=False)

error_df[
    [
        "id",
        "text",
        "gold_triplets",
        "pred_triplets",
        "precision",
        "recall",
        "f1",
        "exact_match",
        "api_error",
    ]
]

In [ ]:
for _, r in error_df.head(10).iterrows():
    print("=" * 120)
    print("ID:", r["id"])
    print("TEXT:", r["text"])
    print("GOLD:", r["gold_triplets"])
    print("PRED:", r["pred_triplets"])
    print("Precision:", r["precision"], "Recall:", r["recall"], "F1:", r["f1"])
    print("API error:", r["api_error"])